### Import packages

In [1]:
%load_ext autoreload
%autoreload 2

#Import general libs you'll need
import os
import geopandas as gpd
from shapely.geometry import Point

# Import functions from main
from utils import readers, analysis, processing
from main import ATL_AGG_CONFIG, ATL08_AGG_CONFIG

# Import functions from the utils
from utils import plotter

### Identify files



In [3]:
atl03_dir = '/home/ejg2736/network_drives/walker/exports/nfs_share/Data/ICESat-2/REL006/florida_aoi/atl03'
atl03_name = 'ATL03_20200212094736_07280607_006_01.h5'
atl03_file = os.path.join(atl03_dir, atl03_name)

# Define ATL08 File
atl08_dir = '/home/ejg2736/network_drives/walker/exports/nfs_share/Data/ICESat-2/REL006/florida_aoi/atl08'
atl08_name = 'ATL08_20200212094736_07280607_006_01.h5'
atl08_file = os.path.join(atl08_dir, atl08_name)

# Define ATL24 file
atl24_dir = '/home/ejg2736/network_drives/walker/exports/nfs_share/Data/ICESat-2/REL006/florida_aoi/atl24'
atl24_name = 'ATL24_20200212094736_07280607_006_01_001_01.h5'
atl24_file = os.path.join(atl24_dir, atl24_name)

# Define the ground track of interest
gt = 'gt1r'

### Read ATL03 File (photon rate data)


In [4]:
# Get photon rate DF
df_ph = readers.read_photon_dataframe(atl03_file, gt, atl08_file)

### Read ATL08 File (Land segment level data)

In [13]:
df_atl08 = readers.read_atl08(atl08_file, gt, atl03_file = atl03_file)

In [14]:
df_atl08.columns

Index(['asr', 'atlas_pa', 'beam_azimuth', 'beam_coelev', 'brightness_flag',
       'can_noise', 'canopy_h_metrics_0', 'canopy_h_metrics_1',
       'canopy_h_metrics_2', 'canopy_h_metrics_3',
       ...
       'photon_rate_te', 'subset_te_flag_0', 'subset_te_flag_1',
       'subset_te_flag_2', 'subset_te_flag_3', 'subset_te_flag_4',
       'terrain_slope', 'terrain_flg', 'urban_flag', 'alongtrack'],
      dtype='object', length=150)

### Read ATL08 (20 m segment)

We can read in an ATL08 file to do this.

In [36]:
df_atl08_20m = readers.read_atl08_20m(atl08_file, gt)

This can also be done by "expanding" an already read in ATL08

In [40]:
df_atl08_20m = readers.expand_atl08_20m_df(df_atl08)

In [42]:
import numpy as np
print(len(np.unique(df_atl08_20m['alongtrack'])))
print(len(df_atl08_20m))

8143
40715


### Reaggregating ATL03 

In [27]:
df_seg = analysis.aggregate_by_segment(df_ph, ATL08_AGG_CONFIG, res=30)

In [31]:
df_seg.columns

Index(['key_id', 'alongtrack', 'latitude', 'longitude', 'solar_elevation',
       'h_te_mean', 'h_te_median', 'h_te_std', 'h_te_min', 'h_te_max',
       'h_canopy', 'h_canopy_max_x', 'h_canopy_mean', 'h_canopy_median',
       'h_canopy_min', 'h_canopy_max_y', 'canopy_openness', 'toc_roughness',
       'canopy_h_metrics', 'h_canopy_abs', 'h_canopy_max_abs', 'n_seg_ph',
       'n_te_photons', 'n_ca_photons', 'n_toc_photons', 'n_shots'],
      dtype='object')

In [45]:
from utils.processing import add_atl08_metadata_to_segments
df_seg = add_atl08_metadata_to_segments(df_seg, df_atl08)

In [32]:
df_seg['photon_rate_can'] = (df_seg['n_ca_photons'] + df_seg['n_toc_photons']) / df_seg['n_shots']
df_seg['photon_rate_te'] = df_seg['n_te_photons'] / df_seg['n_shots']

In [6]:
# Write out as geopandas dataframe
geometry_seg = [Point(xy) for xy in zip(df_seg.longitude, df_seg.latitude)]
gdf_seg = gpd.GeoDataFrame(df_seg, geometry=geometry_seg, crs="EPSG:4326")

# Save geopandas dataframe
gdf_seg.to_file("is2_topobathy.gpkg", layer='atl08atl24', driver="GPKG")